In [170]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
import numpy as np

In [171]:
dataset = fetch_20newsgroups()
X = np.array(dataset.data)
y = np.array(dataset.target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

cuda
(9051,) (2263,) (9051,) (2263,)


In [172]:
model = AutoModel.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5630.77it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [173]:
text = X_train[:8].tolist()
tokenized = tokenizer(text, truncation=True, padding=True, max_length=512, return_tensors="pt")
print(tokenized["input_ids"].shape)

torch.Size([8, 512])


In [ ]:
outs = model(
    input_ids=tokenized["input_ids"],
    attention_mask=tokenized["attention_mask"],
)
embs = outs.last_hidden_state
print(embs.shape)

torch.Size([8, 512, 768])


In [183]:
attention_mask = tokenized["attention_mask"]

# embs: (batch_size, seq_len, hidden_dim) = (8, 512, 768)
# attention_mask: (batch_size, seq_len) = (8, 512)

mask = attention_mask.unsqueeze(-1)          # (8, 512, 1)
masked_embs = embs * mask                    # (8, 512, 768)

token_counts = mask.sum(dim=1)               # (8, 1)
pooled_embeddings = masked_embs.sum(dim=1) / token_counts  # (8, 768)

print(pooled_embeddings.shape)

torch.Size([8, 768])


In [184]:
print(tokenized["attention_mask"].sum(dim=1))

tensor([103, 512, 512, 317, 245, 238, 348, 435])


covered: pretrained text encoder / BERT workflow
dataset: 8 texts from 20 Newsgroups
model: distilbert-base-uncased
artifact: token batch (8, 512), last_hidden_state (8, 512, 768), pooled embeddings (8, 768)
main debug checks: list of strings, truncation to 512, attention-mask mean pooling